In [11]:
# import kagglehub

# from dotenv import load_dotenv

# load_dotenv()

# # Download latest version
# path = kagglehub.competition_download("unipd-deep-learning-2026-challenge-1")

# print("Path to competition files:", path)



In [12]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from timeit import default_timer as timer

import hiddenlayer as hl
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.nn import Conv2d, Linear, MaxPool2d
from torch.utils.data import DataLoader, Dataset, random_split
from torchinfo import summary
from torchview import draw_graph
from torchvision import transforms
from torchvision.transforms import Lambda

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(4242)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cpu


## Dataset and Dataloader



In [42]:
class CelebDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        """
        Args:
            csv_file (str): Path to the csv file with annotations.
            img_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.data_frame = pl.read_csv(
            csv_file,
            schema={
                "id": pl.String,
                "No_Beard": pl.Int8,
                "Young": pl.Int8,
                "Mouth_Slightly_Open": pl.Int8,
                "Smiling": pl.Int8,
                "Male": pl.Int8,
                "Wavy_Hair": pl.Int8,
                "Black_Hair": pl.Int8,
                "Wearing_Hat": pl.Int8,
                "celebrity_id": pl.Int64,
            },
        )
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        # Returns the total number of samples
        return len(self.data_frame)

    def __getitem__(self, idx) -> tuple[torch.Tensor, np.ndarray]:
        # 1. Construct image path and load image
        img_name = os.path.join(self.img_dir, Path(self.data_frame.item(idx, 0) + ".jpg"))
        image = Image.open(img_name).convert('RGB')
        
        # 2. Load labels as float32 for PyTorch compatibility
        labels = np.array(self.data_frame.row(idx)[1:], dtype='float32')

        # 3. Apply the transformation pipeline
        if self.transform:
            image = self.transform(image)
            
        return image, labels

# Define the transformation pipeline
transform = transforms.Compose(
    [
        # Converts a PIL Image (H x W x C) in the range [0, 255] to a torch.FloatTensor (C x H x W) in the range [0.0, 1.0]
        transforms.ToTensor(),
    ]
)

# CHALLENGE_DIR = '/kaggle/input/competitions/unipd-deep-learning-2026-challenge-1'
CHALLENGE_DIR = Path(os.getcwd() + "/data/")

TRAIN_CSV_PATH = f'{CHALLENGE_DIR}/train_data.csv'
TRAIN_IMG_DIR = f'{CHALLENGE_DIR}/train_images'


# Initialize the Dataset and DataLoader
full_train_dataset = CelebDataset(
    csv_file=TRAIN_CSV_PATH,
    img_dir=TRAIN_IMG_DIR,
    transform=transform,
)

# Let's split into training/validation and test sets
train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))

print(f"Data loaded successfully with {len(train_dataset)} training and {len(val_dataset)} validation samples.")

Data loaded successfully with 16000 training and 4000 validation samples.


In [43]:
# Check the shape of the images
image, labels = full_train_dataset[0]

img_shape = image.shape

print("Image shape:\n",
      f"• Channel \t{img_shape[0]} \n",
      f"• Height: \t{img_shape[1]} \n",
      f"• Width: \t{img_shape[2]}\n")

Image shape:
 • Channel 	3 
 • Height: 	60 
 • Width: 	48



## MultiTaskCelebNet

In [ ]:
# Define the Convolutional Neural Network architecture
class MultiTaskCelebNet(nn.Module):
    def __init__(
        self,
        img_shape :tuple = (3, 60, 48), # torch.Size([3, 60, 48])
        conv_filters :list =[],
        kernel_sizes :list =[],
        max_pool_sizes :list =[],
        act_fs :list =[],
        verbose=False,
    ):
        super().__init__()

        assert len(conv_filters) == len(kernel_sizes), (
            "length of {conv_filters} and {kernel_sizes} must be same"
        )
        assert len(conv_filters) == len(max_pool_sizes), (
            "length of {conv_filters} and {max_pool_sizes} must be same"
        )
        # max pool of [1, 1] corresponds to no max pool
        assert len(conv_filters) == len(act_fs), (
            "length of {conv_filters} and {act_fs} must be same"
        )

        self.conv_layers = nn.ModuleList()
        self.max_pools = nn.ModuleList()
        self.in_chan = img_shape[0]  # 3
        self.in_height = img_shape[1]  # 60
        self.in_width = img_shape[2]  # 48
        self.output_dim = len(labels)  # 10
        self.verbose = verbose
        self.act_fs = act_fs

        height_dimension = self.in_height
        width_dimension = self.in_width

        for maxp1, maxp2 in max_pool_sizes:  # as long as padding='same'
            height_dimension = height_dimension // maxp1
            width_dimension = width_dimension // maxp2

        self.inp_dim_to_linear = conv_filters[-1] * height_dimension * width_dimension

        for idx in range(len(conv_filters)):
            if idx == 0:
                # Conv2d(in_channels, out_channels, kernel_size, padding)
                self.conv_layers = self.conv_layers.append(
                    Conv2d(
                        self.in_chan,
                        conv_filters[idx],
                        kernel_sizes[idx],
                        padding="same",
                    )
                )
            else:
                self.conv_layers = self.conv_layers.append(
                    Conv2d(
                        conv_filters[idx - 1],
                        conv_filters[idx],
                        kernel_sizes[idx],
                        padding="same",
                    )
                )

            self.max_pools = self.max_pools.append(MaxPool2d(max_pool_sizes[idx]))

        self.linear = Linear(self.inp_dim_to_linear, self.output_dim)

        pytorch_total_params = sum(
            p.numel() for p in self.parameters() if p.requires_grad
        )
        print(f"My model has {pytorch_total_params} trainable parameters.")

    def forward(self, x):

        for idx in range(len(self.conv_layers)):
            from_shape = x.shape[1:]
            active_conv_layer = self.conv_layers[idx]
            active_max_pool = self.max_pools[idx]
            active_act_fun = self.act_fs[idx]
            x = active_max_pool(active_act_fun(active_conv_layer(x)))
            to_shape = x.shape[1:]

            if self.verbose:
                print(f"From dimension [{from_shape}] to dimension [{to_shape}]")

        x = torch.flatten(
            x, start_dim=1
        )  # if start_dim=1 missed, it also consider batch_size

        if self.verbose:
            print(f"From dimension [{to_shape}] to dimension [{x.shape[1:]}]")
            print(f"From dimension [{x.shape[1:]}] to dimension [{self.output_dim}]")

        return self.linear(x)  # no need to use softmax because of the loss function


Create the model

In [40]:
batch_size = 32

print(f"batch_size: {batch_size}\n")

# DataLoader handles batching, shuffling, and parallel data loading
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(
    f"DataLoader successfully initialized with {len(train_dataset)} training and {len(val_dataset)} validation samples."
)


batch_size: 32

DataLoader successfully initialized with 16000 training and 4000 validation samples.


In [ ]:
# Define model hyperparameters and initialize the model
conv_filters = [32]
kernel_sizes = [[3, 3]]
max_pool_sizes = [[2, 2]]
act_fs = [F.relu]

num_epochs = 20
lr = 1e-3
model = MultiTaskCelebNet(img_shape, conv_filters, kernel_sizes, max_pool_sizes, act_fs, False).to(
    device
)

# Let's visualize the model
model_graph = draw_graph(model, input_size=(batch_size, 3, 32, 32), device=device)
model_graph.visual_graph


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def train(
    model, optimizer, dataloader_train, dataloader_val, epochs, hparam_tuning=False
):
    loss_train, loss_val = [], []
    acc_train, acc_val = [], []
    history1 = hl.History()  # This is a simple tool for logging
    canvas1 = hl.Canvas()  # This is a simple tool for plotting
    for epoch in range(epochs):
        model.train()
        total_acc_train, total_count_train, n_train_batches, total_loss_train = (
            0,
            0,
            0,
            0,
        )
        for idx, (img, label) in enumerate(dataloader_train):
            img, label = img.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(img)
            loss = criterion(logits, label)
            total_loss_train += loss
            loss.backward()
            optimizer.step()

            total_acc_train += (logits.argmax(1) == label).sum().item()
            total_count_train += label.size(0)
            n_train_batches += 1

        avg_loss_train = total_loss_train / n_train_batches
        loss_train.append(avg_loss_train.item())
        accuracy_train = total_acc_train / total_count_train
        acc_train.append(accuracy_train)

        total_acc_val, total_count_val, n_val_batches, total_loss_val = 0, 0, 0, 0
        with torch.no_grad():
            model.eval()
            for idx, (img, label) in enumerate(dataloader_val):
                img, label = img.to(device), label.to(device)
                logits = model(img)
                loss = criterion(logits, label)
                total_loss_val += loss
                total_acc_val += (logits.argmax(1) == label).sum().item()
                total_count_val += label.size(0)
                n_val_batches += 1
        avg_loss_val = total_loss_val / n_val_batches
        loss_val.append(avg_loss_val.item())
        accuracy_val = total_acc_val / total_count_val
        acc_val.append(accuracy_val)
        if epoch % 1 == 0:
            if not hparam_tuning:
                history1.log(
                    epoch,
                    train_loss=avg_loss_train,
                    train_accuracy=accuracy_train,
                    val_loss=avg_loss_val,
                    val_accuracy=accuracy_val,
                )

                with canvas1:
                    canvas1.draw_plot([history1["train_loss"], history1["val_loss"]])
                    canvas1.draw_plot(
                        [history1["train_accuracy"], history1["val_accuracy"]]
                    )
            else:
                print(
                    f"epoch: {epoch + 1} -> Accuracy: {100 * accuracy_train:.2f}%, Loss: {avg_loss_train:.8f}",
                    end=" ---------------- ",
                )
                print(
                    f"Val_Acc: {100 * accuracy_val:.2f}%, Val_Loss: {avg_loss_val:.8f}"
                )
    return loss_train, acc_train, loss_val, acc_val


def plot_learning_acc_and_loss(loss_tr, acc_tr, loss_val, acc_val):
    # Helper function to plot again accuracy and loss

    plt.figure(figsize=(8, 10))

    plt.subplot(2, 1, 1)
    plt.grid()
    plt.plot(range(len(acc_tr)), acc_tr, label="acc_training")
    plt.plot(range(len(acc_tr)), acc_val, label="acc_validation")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend(loc="best")

    plt.subplot(2, 1, 2)
    plt.grid()
    plt.plot(range(len(acc_tr)), loss_tr, label="loss_training")
    plt.plot(range(len(acc_tr)), loss_val, label="loss_validation")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(loc="best")

    plt.show()


In [ ]:
start = timer()
loss_train, accuracy_train, loss_val, accuracy_val = train(
    model, optimizer, dataloader_training, dataloader_validation, epochs=num_epochs
)
end = timer()
print(f"Training time in second: {(end - start)}")

In [ ]:
def test(model, dataloader_test=dataloader_test):
    model.eval()
    total_acc_test, total_count_test, n_batches_test, loss = 0, 0, 0, 0
    for idx, (img, label) in enumerate(dataloader_test):
        img, label = img.to(device), label.to(device)
        logits = model(img)
        loss += criterion(logits, label)
        total_acc_test += (logits.argmax(1) == label).sum().item()
        total_count_test += label.size(0)
        n_batches_test += 1
    accuracy_test = total_acc_test / total_count_test
    loss_test = loss / n_batches_test
    print(f"Test Loss: {loss_test:.8f}", end=" ---------- ")
    print(f"Test Accuracy: {100 * accuracy_test:.4f}%")


In [ ]:
test(model)

Example of submission

In [ ]:
import os
from pathlib import Path
import polars as pl

# CHALLENGE_DIR = '/kaggle/input/competitions/unipd-deep-learning-2026-challenge-1'
CHALLENGE_DIR = Path(
    os.getcwd() + "/data/competitions/unipd-deep-learning-2026-challenge-1"
)

TEST_IMG_DIR = f'{CHALLENGE_DIR}/test_images'
TRAIN_CSV_PATH = f'{CHALLENGE_DIR}/train_data.csv'

# 1. Load the list of test images
test_images = sorted(os.listdir(TEST_IMG_DIR))

# 2. Read only the first row of the train csv to extract column names
# train_df_sample = pd.read_csv(TRAIN_CSV_PATH, nrows=1)
train_df_sample = pl.read_csv(TRAIN_CSV_PATH, n_rows=1)

columns = train_df_sample.columns

# 3. Initialize the submission DataFrame with test IDs and fill the rest of the columns (dummy classification)
submission_df = pl.DataFrame(
    {
        columns[0]: [img.replace(".jpg", "") for img in test_images],
        **{col: [0] * len(test_images) for col in columns[1:]},
    }
)

# 4. Save the output to the working directory (accessible in Kaggle's output panel)
OUTPUT_FILE = Path('submissions/dummy_submission.csv')
submission_df.write_csv(OUTPUT_FILE)#, index=False)

print(f"Dummy submission created successfully: {OUTPUT_FILE}")
print(f"Total rows: {len(submission_df)} | Total columns: {len(columns)}")

In [ ]:
display(submission_df.head())